# Molu Chatbot Inference

Load a saved checkpoint (`model.pt` + `config.json`) and sample responses without re-running the training pipeline.

In [ ]:
from pathlib import Path
import json

import sentencepiece as spm
import torch

CHECKPOINT_DIR = Path("molu_chatbot")
MODEL_PATH = CHECKPOINT_DIR / "model.pt"
CONFIG_PATH = CHECKPOINT_DIR / "config.json"

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing model weights at {MODEL_PATH.resolve()}")
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing config at {CONFIG_PATH.resolve()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

state_dict = torch.load(MODEL_PATH, map_location="cpu")
print(f"Loaded {len(state_dict)} tensors from the checkpoint")
print(f"Config fields: {sorted(config.keys())}")

In [ ]:
import base64
from typing import Any

def load_sentencepiece(config_dict: dict[str, Any], checkpoint_dir: Path) -> spm.SentencePieceProcessor:
    tokenizer_info = config_dict.get("tokenizer") or {}
    processor = spm.SentencePieceProcessor()
    source = None

    if "serialized_proto" in tokenizer_info:
        serialized = tokenizer_info["serialized_proto"]
        if isinstance(serialized, str):
            serialized_bytes = bytes.fromhex(serialized)
        else:
            serialized_bytes = bytes(serialized)
        processor.LoadFromSerializedProto(serialized_bytes)
        source = "config.serialized_proto"
    elif "model_base64" in tokenizer_info:
        serialized_bytes = base64.b64decode(tokenizer_info["model_base64"])
        processor.LoadFromSerializedProto(serialized_bytes)
        source = "config.model_base64"
    elif "model_bytes" in tokenizer_info:
        serialized_bytes = bytes(tokenizer_info["model_bytes"])
        processor.LoadFromSerializedProto(serialized_bytes)
        source = "config.model_bytes"
    elif "model_path" in tokenizer_info:
        model_path = Path(tokenizer_info["model_path"])
        if not model_path.is_file():
            model_path = checkpoint_dir / model_path
        if not model_path.is_file():
            raise FileNotFoundError(f"SentencePiece file not found at {model_path}")
        processor.Load(str(model_path))
        source = str(model_path.resolve())
    else:
        candidates = sorted(checkpoint_dir.glob("*.model"))
        if not candidates:
            raise FileNotFoundError(
                "Could not locate a SentencePiece model. Embed it under config['tokenizer'] "
                "or place a *.model file next to model.pt."
            )
        processor.Load(str(candidates[0]))
        source = str(candidates[0].resolve())

    print(f"Loaded SentencePiece model from {source}")
    return processor

sp = load_sentencepiece(config, CHECKPOINT_DIR)

special_tokens = config.get("special_tokens") or {}
PAD = int(special_tokens.get("PAD", sp.pad_id()))
BOS = int(special_tokens.get("BOS", sp.bos_id()))
sep_piece_id = sp.piece_to_id("<sep>")
SEP = int(special_tokens.get("SEP", sep_piece_id if sep_piece_id != -1 else sp.bos_id()))
EOS = int(special_tokens.get("EOS", sp.eos_id()))

VOCAB_SIZE = int(config.get("vocab_size", state_dict["tok_emb.weight"].shape[0]))
MAX_LEN = int(config.get("max_len", state_dict["pos_emb.weight"].shape[0]))

def encode_text(text: str) -> list[int]:
    return sp.encode(text, out_type=int)

def decode_text(token_ids) -> str:
    drop = {PAD, BOS, SEP, EOS}
    filtered = [int(tid) for tid in token_ids if int(tid) not in drop]
    return sp.decode(filtered)

print(f"Special tokens -> PAD: {PAD}, BOS: {BOS}, SEP: {SEP}, EOS: {EOS}")
print(f"Configured MAX_LEN: {MAX_LEN}")

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.dk = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        bsz, seq_len, hidden = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(bsz, seq_len, self.n_heads, self.dk).transpose(1, 2)
        k = k.view(bsz, seq_len, self.n_heads, self.dk).transpose(1, 2)
        v = v.view(bsz, seq_len, self.n_heads, self.dk).transpose(1, 2)

        attn_bias = None
        if attn_mask is not None:
            if attn_mask.dim() != 2 or attn_mask.shape != (bsz, seq_len):
                raise ValueError("attn_mask must be (batch, seq_len)")
            pad_mask = (attn_mask == 0).unsqueeze(1).unsqueeze(2)
            neg_inf = torch.finfo(q.dtype).min
            attn_bias = pad_mask.to(dtype=q.dtype) * neg_inf

        dropout_p = self.attn_drop.p if self.training else 0.0
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_bias,
            dropout_p=dropout_p,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(bsz, seq_len, hidden)
        return self.resid_drop(self.proj(y))

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp_ratio: int = 4, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_ratio * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask=None):
        x = x + self.attn(self.ln1(x), attn_mask)
        x = x + self.mlp(self.ln2(x))
        return x

class GPTScratch(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_layers: int,
        n_heads: int,
        max_len: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, 4, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)
        self.head.weight = self.tok_emb.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x, attention_mask=None):
        batch, seq_len = x.shape
        if seq_len > self.max_len:
            x = x[:, -self.max_len :]
            seq_len = x.shape[1]
            if attention_mask is not None:
                attention_mask = attention_mask[:, -self.max_len :]
        pos = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
        h = self.tok_emb(x) + self.pos_emb(pos)
        h = self.drop(self.emb_norm(h))
        for block in self.blocks:
            h = block(h, attention_mask)
        h = self.ln_f(h)
        return self.head(h)

In [ ]:
import re

def infer_architecture(state_dict):
    d_model = state_dict["tok_emb.weight"].shape[1]
    vocab_size = state_dict["tok_emb.weight"].shape[0]
    max_len = state_dict["pos_emb.weight"].shape[0]
    layer_pattern = re.compile("blocks\.(\d+)\.")
    layers = {int(match.group(1)) for key in state_dict for match in [layer_pattern.match(key)] if match}
    n_layers = max(layers) + 1 if layers else int(config.get("n_layers", 1))
    return vocab_size, d_model, max_len, n_layers

vocab_sd, d_model_sd, max_len_sd, n_layers_sd = infer_architecture(state_dict)

def pick_head_count(d_model: int, preferred: int | None) -> int:
    divisors = [h for h in range(1, 65) if d_model % h == 0]
    if not divisors:
        raise ValueError(f"d_model={d_model} has no valid head count")
    if preferred in divisors:
        return int(preferred)
    target = preferred or divisors[0]
    return min(divisors, key=lambda h: abs(h - target))

n_heads = pick_head_count(d_model_sd, config.get("n_heads"))

model = GPTScratch(
    vocab_size=vocab_sd,
    d_model=d_model_sd,
    n_layers=n_layers_sd,
    n_heads=n_heads,
    max_len=max_len_sd,
    dropout=float(config.get("dropout", 0.1)),
).to(device)

missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
if missing_keys:
    print(f"Missing keys: {missing_keys}")
if unexpected_keys:
    print(f"Unexpected keys: {unexpected_keys}")

model.eval()
MODEL_MAX_LEN = max_len_sd
MAX_LEN = MODEL_MAX_LEN
print(
    f"Restored model with vocab={vocab_sd}, d_model={d_model_sd}, "
    f"layers={n_layers_sd}, heads={n_heads}, max_len={MODEL_MAX_LEN}"
)

In [ ]:
import torch

HIST_MAX = MODEL_MAX_LEN
MAX_NEW_TOKENS = int(config.get("generation", {}).get("max_new_tokens", 96))

@torch.no_grad()
def sample_top_p(logits_row: torch.Tensor, top_p: float = 0.9, temperature: float = 0.7) -> int:
    logits_row = logits_row / max(1e-6, temperature)
    probs = torch.softmax(logits_row, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    mask = cumulative > top_p
    if mask.any():
        mask[..., 0] = False
    clipped = sorted_probs.masked_fill(mask, 0.0)
    if clipped.sum() <= 0:
        return int(torch.argmax(probs).item())
    clipped = clipped / clipped.sum()
    sampled_idx = torch.multinomial(clipped, 1)
    return int(sorted_idx[sampled_idx])

@torch.no_grad()
def generate_reply(
    prompt: str,
    max_ctx: int = HIST_MAX,
    max_new_tokens: int = MAX_NEW_TOKENS,
    top_p: float = 0.9,
    temperature: float = 0.7,
) -> str:
    tokens = [BOS] + encode_text(prompt) + [SEP]
    tokens = tokens[-max_ctx:]
    context = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_tokens):
        logits = model(context)
        next_token = sample_top_p(logits[0, -1], top_p=top_p, temperature=temperature)
        tokens.append(next_token)
        context = torch.tensor(tokens[-max_ctx:], dtype=torch.long, device=device).unsqueeze(0)
        if next_token == EOS:
            break
    try:
        last_sep = len(tokens) - 1 - tokens[::-1].index(SEP)
    except ValueError:
        last_sep = 0
    try:
        last_eos = len(tokens) - 1 - tokens[::-1].index(EOS)
    except ValueError:
        last_eos = len(tokens)
    return decode_text(tokens[last_sep:last_eos]).strip()

def batch_generate(prompts, **kwargs):
    outputs = []
    for prompt in prompts:
        outputs.append({"prompt": prompt, "reply": generate_reply(prompt, **kwargs)})
    return outputs

In [ ]:
sample_prompt = "넷플릭스에서 요즘 뭐 봐?"
print(f"Prompt: {sample_prompt}")
print(f"Reply: {generate_reply(sample_prompt)}")